# Train the Jarvie wake/sleep models (Google Colab)

Trains two livekit-wakeword classifiers, `jarvie_wake` and `jarvie_sleep`, on a Colab GPU and saves the
`.onnx` files to Google Drive. Evaluate them afterwards on your own recordings with
`server/jarvie/eval/wakeword_eval.ipynb` (on the Mac).

**Why the cells are laid out this way:** livekit-wakeword imports many training packages only when a
stage starts, so a missing package used to surface 20+ minutes into a run. Here everything is installed
and checked up front (cells 1-2), all data is downloaded once per session (cell 3), and a tiny smoke run
exercises every stage in a few minutes (cell 6) before the real run (cell 7).

**Before you start:** Runtime > Change runtime type > **T4 GPU** (or better).

**After a disconnect:** re-run cells 2, 3, 4, 5, then 7. Finished stages are skipped (a marker file is
written to Drive after each stage). Note that livekit itself only resumes the *generation* stage; a
training run that was cut off mid-way starts that stage over.

## 1. Install everything (once per session)
Installs the system programs and Python packages every stage needs. If pip asks you to restart the session, do it once (Runtime > Restart session) and continue with cell 2; don't re-run this cell.

In [ ]:
# System programs (not pip packages):
#   espeak-ng: Piper turns text into phonemes by calling the espeak-ng command (run_generate)
#   ffmpeg:    decodes m4a/mp3 room recordings for experiment C (usually preinstalled on Colab)
!apt-get -qq update && apt-get -qq install -y espeak-ng ffmpeg > /dev/null

# Python packages:
#   [train]: torch, librosa, scipy, audiomentations, webrtcvad, nltk, pronouncing, cmudict, ... (imported mid-run)
#   [eval]:  matplotlib, imported only by run_eval at the very end
#   onnx, onnxscript: run_export uses torch.onnx's exporter (needs onnxscript) and onnx to clean the file;
#                     neither is in livekit's extras
!pip install -q "livekit-wakeword[train,eval]" "setuptools<81" onnx onnxscript

!espeak-ng --version

## 2. Preflight check
Fails in seconds if anything is missing, instead of mid-run. Continue only when it prints `missing: none` and a GPU name.

In [ ]:
import importlib
import os
import shutil

# Must be set before torch first uses the GPU: lets PyTorch grow memory in pieces instead of needing
# one big free block, which avoids the "CUDA out of memory ... Skipping 50 clips" failures.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import nltk  # noqa: E402
import torch  # noqa: E402

modules = [
    "torch", "torchaudio", "soundfile", "librosa", "scipy.signal", "audiomentations",
    "torch_audiomentations", "webrtcvad", "nltk", "pronouncing", "cmudict", "sklearn",
    "tqdm", "yaml", "rich", "huggingface_hub", "matplotlib", "onnxruntime",
    "onnxruntime.quantization", "onnx", "onnxscript", "livekit.wakeword",
]
missing = []
for m in modules:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"{m}: {e}")

for program in ["espeak-ng", "ffmpeg"]:
    if shutil.which(program) is None:
        missing.append(f"{program}: program not found (apt-get install {program})")

nltk.download("cmudict", quiet=True)  # livekit would otherwise download this mid-run

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > GPU")
print("missing:", missing or "none")

## 3. Paths and training data (once per session)
- `DATA_DIR` is on Colab's **local disk**: the ACAV100M background-speech features are ~16 GB, more than a free
  Drive holds, so they're re-downloaded each session (takes a while).
- `OUTPUT_DIR` is on **Drive**, so generated clips, features, checkpoints and the exported models survive
  disconnects.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/wakeword/output")
SMOKE_DIR = Path("/content/smoke")  # throwaway output for the smoke test
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Speech-synthesis voice weights, ACAV100M features (~16 GB), room reverbs, background noise
!livekit-wakeword setup --data-dir {DATA_DIR}

## 4. Phrases, negatives and training settings
- Every phrase stays under ~2 s even when spoken slowly: livekit trains on 2.0 s windows and crops the *start*
  of longer clips, so "jarvie, I need some help" (slow) would lose "jarvie". It's replaced by "jarvie, help me".
- `custom_negative_phrases`: livekit's automatic hard negatives only swap sounds inside single words, so the
  phrases *without* "jarvie" ("wake up", "help me") must be listed explicitly. Each model also gets the other
  model's phrases, so "jarvie, go to sleep" can't wake the bot and vice versa.
- `length_scales`: speaking rate (smaller = faster), widened for fast/slow speech, capped at 1.2 so slow
  phrases still fit in 2 s.

In [ ]:
from livekit.wakeword import WakeWordConfig

PHRASES = {
    "jarvie_wake": [
        "jarvie, wake up",
        "jarvie, help me",
        "jarvie, wakie wakie",
        "jarvee, wake up",  # spelling variant; keep only if the TTS says "jarvie" wrong (listen in cell 6)
    ],
    "jarvie_sleep": [
        "jarvie, go to sleep",
        "jarvie, sleep now",
        "jarvee, go to sleep",
    ],
}

NEGATIVES = {
    "jarvie_wake": [
        "wake up", "wake up wake up", "wakie wakie", "help me", "can you help me",
        "i need some help", "jarvie", "jacky wake up", "harvey wake up", "jarvis wake up",
        "barbie help me", "carvey wakie wakie",
    ],
    "jarvie_sleep": [
        "sleep", "go to sleep", "time to sleep", "i need some sleep", "jarvie",
        "jarvie keep", "jarvie sweep", "jacky go to sleep", "harvey go to sleep",
    ],
}
# Cross-negatives: each model must ignore the other model's phrases.
NEGATIVES["jarvie_wake"] += PHRASES["jarvie_sleep"]
NEGATIVES["jarvie_sleep"] += PHRASES["jarvie_wake"]

FULL = dict(
    tts_batch_size=25,  # default 50 occasionally runs a T4 (15 GB) out of memory
    n_samples=20000,
    n_samples_val=4000,
    n_background_samples=2000,
    n_background_samples_val=500,
    steps=80000,
    length_scales=[0.65, 0.75, 0.85, 1.0, 1.1, 1.2],
    augmentation={"rounds": 3},
    model={"model_size": "medium"},  # use "small" for the Raspberry Pi build
    max_negative_weight=3000,
    target_fp_per_hour=0.1,
)

# Same settings as FULL (model size, augmentation rounds, speeds, negatives), tiny sizes:
# runs every stage in a few minutes, with the real configuration, to catch errors early.
SMOKE = dict(
    FULL,
    n_samples=20,
    n_samples_val=10,
    n_background_samples=10,
    n_background_samples_val=5,
    steps=200,
)


def make_configs(settings: dict, output_dir: Path, prefix: str = "") -> dict[str, WakeWordConfig]:
    return {
        name: WakeWordConfig(
            model_name=f"{prefix}{name}",
            target_phrases=phrases,
            custom_negative_phrases=NEGATIVES[name],
            data_dir=str(DATA_DIR),
            output_dir=str(output_dir),
            **settings,
        )
        for name, phrases in PHRASES.items()
    }

## 5. Pipeline helper
Runs the six stages for one model. After each stage it writes a marker file to `OUTPUT_DIR/<model>/_done/`,
and skips that stage on re-runs, so a disconnect only costs the stage that was running.

In [ ]:
import json
import os

import onnxruntime as ort
from livekit.wakeword import (
    run_augment,
    run_eval,
    run_export,
    run_extraction,
    run_generate,
    run_train,
)

sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = os.cpu_count() or 2


def train_model(cfg: WakeWordConfig, output_dir: Path) -> dict:
    done_dir = output_dir / cfg.model_name / "_done"
    done_dir.mkdir(parents=True, exist_ok=True)

    def stage(name, fn, *args):
        marker = done_dir / f"{name}.json"
        if marker.exists():
            print(f"  {name}: already done, skipping")
            return json.loads(marker.read_text())
        print(f"  {name}: running")
        result = fn(cfg, *args)
        marker.write_text(json.dumps(result, default=str))
        return result

    stage("generate", run_generate)
    stage("augment", run_augment)
    stage("extract", run_extraction, sess_options)
    stage("train", run_train)
    onnx_path = stage("export", run_export)
    metrics = stage("eval", run_eval, onnx_path)
    return {"onnx": str(onnx_path), **metrics}

## 6. Smoke test (a few minutes)
Runs every stage on tiny data in a throwaway folder. If anything is still missing, it fails here: add it to cell 1,
then re-run cells 2 and 6. Also **listen to a couple of generated positive clips** to check how the TTS says
"jarvie"; if it sounds wrong, keep the "jarvee" spelling variants in cell 4.

In [ ]:
import shutil

shutil.rmtree(SMOKE_DIR, ignore_errors=True)  # always start the smoke test clean
for name, cfg in make_configs(SMOKE, SMOKE_DIR, prefix="smoke_").items():
    print(f"=== smoke: {name} ===")
    print(train_model(cfg, SMOKE_DIR))
print("smoke test passed")

In [ ]:
# Listen to a few synthetic positives from the smoke run.
from IPython.display import Audio, display

clips = sorted(SMOKE_DIR.glob("smoke_jarvie_wake/**/positive_train/*.wav"))[:3]
print(clips or "no clips found - check the folder layout under", SMOKE_DIR)
for c in clips:
    display(Audio(str(c)))

## 7. Full training run (hours)
Re-run this cell after a disconnect (after cells 2-5); finished stages are skipped. `run_eval` reports recall and
false accepts per hour on synthetic test speech and livekit's background audio; the real test is your own
recordings in `wakeword_eval.ipynb`.

In [ ]:
results = {}
for name, cfg in make_configs(FULL, OUTPUT_DIR).items():
    print(f"=== {name} ===")
    results[name] = r = train_model(cfg, OUTPUT_DIR)
    print(f"{name}: AUT={r['aut']:.4f}  FPPH={r['fpph']:.2f}  Recall={r['recall']:.1%}  "
          f"suggested threshold={r.get('optimal_threshold', r.get('threshold'))}")

(OUTPUT_DIR / "results.json").write_text(json.dumps(results, indent=2, default=str))
results

## 8. Get the models
The `.onnx` files are already on Drive (paths printed below). Download them, put them in `server/models/` on the Mac
(`jarvie_wake.onnx`, `jarvie_sleep.onnx`), then restart the eval notebook's kernel and Run All.

In [ ]:
from google.colab import files

for name, r in results.items():
    print(name, "->", r["onnx"])
    files.download(r["onnx"])

## 9. Augmentation experiments: B (realistic) and C (your room)
Compares livekit's default augmentation (**A**, the run above) with two gentler variants. Judge them on your
real recordings in `wakeword_eval.ipynb`, not on `run_eval`'s synthetic numbers.

| | noise | noise level (SNR) | reverb | clipping | passes | background audio |
|---|---|---|---|---|---|---|
| A | every clip | 5-15 dB | 50%, louder | yes | per `FULL` | MUSAN |
| B | 70% of clips | 10-25 dB | 30%, same loudness | prevented | 1 | MUSAN |
| C | 70% of clips | 10-25 dB | 30%, same loudness | prevented | 1 | **your room** |

B and C reuse A's generated speech (the slowest stage), so run them **after A's generation stage has
finished**. They only redo augmentation, feature extraction and training, into separate folders.

**In a new session:** run cells 2-5 first. This section's patch is only active inside `run_experiment`; it
restores livekit's original augmentation afterwards, so re-running A is unaffected.

In [ ]:
# Patch livekit's augmentation with adjustable settings. run_augment() builds an AudioAugmentor and
# calls these two methods for every clip, so replacing them on the class changes every round.
import random
import re
import shutil

import numpy as np
from livekit.wakeword.data import augment as lk_aug

# Keep the true originals even if this cell is run twice.
if "_ORIGINAL_AUG" not in globals():
    _ORIGINAL_AUG = {
        "apply_rir": lk_aug.AudioAugmentor.apply_rir,
        "mix_with_background": lk_aug.AudioAugmentor.mix_with_background,
    }

AUG_PRESETS = {
    "A": None,  # livekit's own augmentation, untouched
    "B": {"noise_prob": 0.7, "snr_db": (10.0, 25.0), "rir_prob": 0.3, "max_peak": 0.95},
    "C": {"noise_prob": 0.7, "snr_db": (10.0, 25.0), "rir_prob": 0.3, "max_peak": 0.95},
}
_active_aug: dict = {}


def _rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x)))) if x.size else 0.0


def _apply_rir_same_loudness(self, audio, p=0.5):
    """Reverb with probability rir_prob, rescaled to the input's loudness (livekit's isn't)."""
    out = _ORIGINAL_AUG["apply_rir"](self, audio, p=_active_aug["rir_prob"])
    if out is audio:  # reverb skipped this time
        return audio
    rms_in, rms_out = _rms(audio), _rms(out)
    return (out * (rms_in / rms_out)).astype(np.float32) if rms_out > 0 else out


def _mix_sometimes_no_clipping(self, audio, snr_db_range=(5.0, 15.0)):
    """Noise on noise_prob of clips at a gentler SNR; then scale down anything that would clip.

    This is the last step before livekit writes the clip as 16-bit WAV, which hard-clips above 1.0.
    """
    if random.random() < _active_aug["noise_prob"]:
        audio = _ORIGINAL_AUG["mix_with_background"](self, audio, snr_db_range=_active_aug["snr_db"])
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > _active_aug["max_peak"]:
        audio = audio * (_active_aug["max_peak"] / peak)
    return np.asarray(audio, dtype=np.float32)


def use_augmentation(name: str) -> None:
    preset = AUG_PRESETS[name]
    if preset is None:
        lk_aug.AudioAugmentor.apply_rir = _ORIGINAL_AUG["apply_rir"]
        lk_aug.AudioAugmentor.mix_with_background = _ORIGINAL_AUG["mix_with_background"]
    else:
        _active_aug.clear()
        _active_aug.update(preset)
        lk_aug.AudioAugmentor.apply_rir = _apply_rir_same_loudness
        lk_aug.AudioAugmentor.mix_with_background = _mix_sometimes_no_clipping
    print(f"augmentation {name}: {preset or 'livekit default'}")


use_augmentation("A")  # nothing changes until an experiment asks for it

### Your room's background audio (for C)
Record 20-30 minutes of your dorm with the microphone the detector will use: silence, fan/AC, music at a normal
volume, a roommate talking, a video playing. Several files are fine (any format: wav, m4a, mp3, flac). Put them in
`MyDrive/wakeword/my_noise/`. They're converted to 16 kHz mono here, because livekit reads background files
**without resampling**: a 44.1 kHz recording would otherwise be played back at the wrong speed and pitch.

In [ ]:
import librosa
import soundfile as sf

MY_NOISE_DIR = Path("/content/drive/MyDrive/wakeword/my_noise")  # your uploads
MY_NOISE_16K = Path("/content/my_noise_16k")  # converted copies (local disk)


def prepare_my_noise() -> float:
    files = sorted(
        p for p in MY_NOISE_DIR.rglob("*") if p.suffix.lower() in {".wav", ".flac", ".mp3", ".m4a", ".ogg"}
    )
    if not files:
        raise SystemExit(f"no recordings found in {MY_NOISE_DIR}")
    MY_NOISE_16K.mkdir(parents=True, exist_ok=True)
    total_s = 0.0
    for f in files:
        y, _ = librosa.load(str(f), sr=16000, mono=True)
        sf.write(MY_NOISE_16K / f"{f.stem}.wav", y, 16000, subtype="FLOAT")
        total_s += len(y) / 16000
    print(f"{len(files)} file(s), {total_s / 60:.1f} min of room audio -> {MY_NOISE_16K}")
    return total_s


prepare_my_noise()

In [ ]:
# Speech generation is identical for A, B and C, so copy A's clean generated clips (clip_000000.wav, ...)
# instead of regenerating them; livekit then sees each split as complete and skips generation.
SPLITS = ["positive_train", "positive_test", "negative_train", "negative_test",
          "background_train", "background_test"]
CLEAN_CLIP = re.compile(r"^clip_\d{6}\.wav$")

# Must match the settings run A was trained with (sizes, speeds, model); only augmentation differs.
EXPERIMENT_SETTINGS = dict(FULL)


def reuse_generated_clips(src_root: Path, dst_root: Path, model: str) -> None:
    src, dst = src_root / model, dst_root / model
    if not (src / "_done" / "generate.json").exists():
        raise RuntimeError(f"run A hasn't finished generating {model} yet ({src})")
    copied = 0
    for split in SPLITS:
        if not (src / split).exists():
            continue
        (dst / split).mkdir(parents=True, exist_ok=True)
        for f in (src / split).iterdir():
            if CLEAN_CLIP.match(f.name) and not (dst / split / f.name).exists():
                shutil.copy2(f, dst / split / f.name)
                copied += 1
    print(f"  {model}: reused {copied} generated clips from run A")


def run_experiment(name: str) -> dict:
    out_dir = OUTPUT_DIR.parent / f"exp_{name}"
    augmentation = {**EXPERIMENT_SETTINGS.get("augmentation", {}), "rounds": 1}
    if name == "C":
        augmentation["background_paths"] = [str(MY_NOISE_16K)]
    settings = dict(EXPERIMENT_SETTINGS, augmentation=augmentation)

    use_augmentation(name)
    try:
        results = {}
        for model, cfg in make_configs(settings, out_dir).items():
            print(f"=== experiment {name}: {model} ===")
            reuse_generated_clips(OUTPUT_DIR, out_dir, model)
            results[model] = train_model(cfg, out_dir)
        (out_dir / "results.json").write_text(json.dumps(results, indent=2, default=str))
        return results
    finally:
        use_augmentation("A")  # always restore livekit's original augmentation

In [ ]:
# Experiment B (hours). Re-run after a disconnect: finished stages are skipped.
exp_results = {"A": results} if "results" in globals() else {}
exp_results["B"] = run_experiment("B")

In [ ]:
# Experiment C (hours). Needs prepare_my_noise() above to have run in this session.
exp_results["C"] = run_experiment("C")

In [ ]:
# run_eval numbers (synthetic speech + livekit background audio), side by side.
import pandas as pd

rows = []
for variant, per_model in exp_results.items():
    for model, r in per_model.items():
        rows.append({"variant": variant, "model": model, "aut": r["aut"], "fpph": r["fpph"],
                     "recall": r["recall"], "suggested_threshold": r.get("optimal_threshold")})
pd.DataFrame(rows).set_index(["model", "variant"]).sort_index()

In [ ]:
# Download every variant as <model>_<variant>.onnx. On the Mac, copy the one to test into
# server/models/ as <model>.onnx (e.g. jarvie_wake_B.onnx -> jarvie_wake.onnx), restart the eval
# notebook's kernel, Run All, and note normal-pace EER and how many "similar" negatives fire.
from google.colab import files

for variant, per_model in exp_results.items():
    for model, r in per_model.items():
        named = Path("/content") / f"{model}_{variant}.onnx"
        shutil.copy2(r["onnx"], named)
        files.download(str(named))

## Optional: free Drive space
Training only reads the extracted features, so once a model's `extract` stage is done you can delete its
augmented WAV files. Check usage first.

In [ ]:
!du -sh {OUTPUT_DIR}/*